# Nonlinear Filters --- EKF, UKF, EnKF

This notebook demonstrates the three nonlinear filters available in **TFiltersPy**:

| Filter | Key Idea | When to Use |
|--------|----------|-------------|
| **ExtendedKalmanFilter** | Linearizes via Jacobians | Low-dimensional, differentiable dynamics |
| **UnscentedKalmanFilter** | Sigma-point propagation | Low-dimensional, hard-to-differentiate dynamics |
| **EnsembleKalmanFilter** | Monte Carlo ensemble | High-dimensional systems (50+ states) |

All three share the same sklearn-compatible API: `fit()`, `predict()`, `score()`.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from tfilterspy import ExtendedKalmanFilter, UnscentedKalmanFilter, EnsembleKalmanFilter

np.random.seed(42)

---
## Section 1: Pendulum Tracking --- EKF vs UKF

A nonlinear pendulum has state $\mathbf{x} = [\theta, \omega]^T$ (angle and angular velocity).
The dynamics include `sin(theta)`, making this a classically nonlinear problem.
We observe `sin(theta)` through a noisy sensor.

We compare the **EKF** (which needs hand-coded Jacobians) with the **UKF** (which does not).

In [ ]:
# --- Pendulum dynamics ---
dt = 0.01

def f(x):
    """State transition: x = [theta, omega]"""
    theta, omega = x
    return np.array([theta + dt * omega, omega - dt * 9.81 * np.sin(theta)])

def h(x):
    """Observation: we measure sin(theta)"""
    return np.array([np.sin(x[0])])

def F_jac(x):
    """Jacobian of f w.r.t. x"""
    theta, omega = x
    return np.array([[1, dt],
                     [-dt * 9.81 * np.cos(theta), 1]])

def H_jac(x):
    """Jacobian of h w.r.t. x"""
    return np.array([[np.cos(x[0]), 0]])

In [ ]:
# --- Generate ground truth and noisy observations ---
N = 500
Q = np.diag([0.001, 0.01])
R = np.array([[0.1]])

true_states = np.zeros((N, 2))
measurements = np.zeros((N, 1))

true_states[0] = [0.5, 0.0]  # initial angle = 0.5 rad

for k in range(1, N):
    true_states[k] = f(true_states[k - 1]) + np.random.multivariate_normal([0, 0], Q)

for k in range(N):
    measurements[k] = h(true_states[k]) + np.random.multivariate_normal([0], R)

print(f"Generated {N} timesteps.")
print(f"True theta range: [{true_states[:, 0].min():.2f}, {true_states[:, 0].max():.2f}] rad")

In [ ]:
# --- Fit EKF and UKF ---
x0 = np.array([0.1, 0.0])
P0 = np.eye(2) * 0.1

ekf = ExtendedKalmanFilter(f, h, F_jac, H_jac, Q, R, x0, P0)
ukf = UnscentedKalmanFilter(f, h, Q, R, x0, P0, alpha=1e-3, beta=2.0, kappa=0)

ekf.fit(measurements)
ukf.fit(measurements)

ekf_states = ekf.predict()
ukf_states = ukf.predict()

# --- RMSE ---
rmse_ekf_theta = np.sqrt(np.mean((ekf_states[:, 0] - true_states[:, 0]) ** 2))
rmse_ukf_theta = np.sqrt(np.mean((ukf_states[:, 0] - true_states[:, 0]) ** 2))
rmse_ekf_omega = np.sqrt(np.mean((ekf_states[:, 1] - true_states[:, 1]) ** 2))
rmse_ukf_omega = np.sqrt(np.mean((ukf_states[:, 1] - true_states[:, 1]) ** 2))

print(f"{'Filter':<8} {'RMSE theta':>12} {'RMSE omega':>12} {'score()':>10}")
print("-" * 44)
print(f"{'EKF':<8} {rmse_ekf_theta:>12.5f} {rmse_ekf_omega:>12.5f} {ekf.score(true_states):>10.5f}")
print(f"{'UKF':<8} {rmse_ukf_theta:>12.5f} {rmse_ukf_omega:>12.5f} {ukf.score(true_states):>10.5f}")

In [ ]:
# --- Plot theta estimates ---
t = np.arange(N) * dt

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(t, true_states[:, 0], "k-", lw=1.5, label="True theta")
axes[0].plot(t, ekf_states[:, 0], "b--", lw=1, alpha=0.8, label="EKF")
axes[0].plot(t, ukf_states[:, 0], "r:", lw=1, alpha=0.8, label="UKF")
axes[0].set_ylabel("theta (rad)")
axes[0].set_title("Pendulum Tracking: theta")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, true_states[:, 1], "k-", lw=1.5, label="True omega")
axes[1].plot(t, ekf_states[:, 1], "b--", lw=1, alpha=0.8, label="EKF")
axes[1].plot(t, ukf_states[:, 1], "r:", lw=1, alpha=0.8, label="UKF")
axes[1].set_ylabel("omega (rad/s)")
axes[1].set_xlabel("Time (s)")
axes[1].set_title("Pendulum Tracking: omega")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Section 2: Radar Target Tracking --- EKF with Smoothing

An aircraft flies a **coordinated turn** (constant turn rate $\omega = 0.02$ rad/s).
A radar station at the origin measures **range** and **bearing** --- both nonlinear
functions of the Cartesian state $\mathbf{x} = [x, v_x, y, v_y]^T$.

After filtering, we apply the **RTS smoother** (`ekf.smooth()`) to show how
smoothing reduces estimation error by using future measurements.

In [ ]:
# --- Aircraft dynamics (coordinated turn) ---
DT = 0.5
OMEGA = 0.02  # turn rate (rad/s)

def f_aircraft(x):
    """Constant-velocity model (filter assumes straight-line motion)."""
    return np.array([x[0] + DT * x[1],
                     x[1],
                     x[2] + DT * x[3],
                     x[3]])

def h_radar(x):
    """Radar measures [range, bearing]."""
    r = np.sqrt(x[0]**2 + x[2]**2)
    theta = np.arctan2(x[2], x[0])
    return np.array([r, theta])

def F_aircraft_jac(x):
    return np.array([[1, DT, 0, 0],
                     [0, 1,  0, 0],
                     [0, 0,  1, DT],
                     [0, 0,  0, 1]])

def H_radar_jac(x):
    px, vx, py, vy = x
    r = np.sqrt(px**2 + py**2)
    if r < 1e-10:
        r = 1e-10
    return np.array([[px / r, 0, py / r, 0],
                     [-py / r**2, 0, px / r**2, 0]])

In [ ]:
# --- Generate curved trajectory (coordinated turn) ---
N2 = 500

true_aircraft = np.zeros((N2, 4))  # [x, vx, y, vy]
true_aircraft[0] = [1000, 50, 500, 20]  # initial position and velocity

for k in range(1, N2):
    px, vx, py, vy = true_aircraft[k - 1]
    speed = np.sqrt(vx**2 + vy**2)
    heading = np.arctan2(vy, vx) + OMEGA * DT
    vx_new = speed * np.cos(heading)
    vy_new = speed * np.sin(heading)
    true_aircraft[k] = [px + DT * vx_new, vx_new, py + DT * vy_new, vy_new]

# Noisy radar measurements
range_std = 50.0    # meters
bearing_std = 0.02  # radians
R2 = np.diag([range_std**2, bearing_std**2])

radar_meas = np.zeros((N2, 2))
for k in range(N2):
    radar_meas[k] = h_radar(true_aircraft[k]) + np.array([
        np.random.randn() * range_std,
        np.random.randn() * bearing_std
    ])

print(f"Trajectory spans x=[{true_aircraft[:, 0].min():.0f}, {true_aircraft[:, 0].max():.0f}] m")
print(f"                 y=[{true_aircraft[:, 2].min():.0f}, {true_aircraft[:, 2].max():.0f}] m")

In [ ]:
# --- Fit EKF, then smooth ---
Q2 = np.diag([1, 0.1, 1, 0.1])  # process noise
x0_ac = np.array([1000, 50, 500, 20])
P0_ac = np.diag([100, 10, 100, 10])

ekf_ac = ExtendedKalmanFilter(f_aircraft, h_radar, F_aircraft_jac, H_radar_jac,
                              Q2, R2, x0_ac, P0_ac)
ekf_ac.fit(radar_meas)
filtered = ekf_ac.predict()
smoothed, smoothed_covs = ekf_ac.smooth()

print("Filtering and smoothing complete.")

In [ ]:
# --- Convert radar measurements to Cartesian for plotting ---
meas_x = radar_meas[:, 0] * np.cos(radar_meas[:, 1])
meas_y = radar_meas[:, 0] * np.sin(radar_meas[:, 1])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Trajectory plot
axes[0].plot(meas_x, meas_y, ".", color="gray", ms=2, alpha=0.4, label="Measurements")
axes[0].plot(true_aircraft[:, 0], true_aircraft[:, 2], "k-", lw=2, label="True")
axes[0].plot(filtered[:, 0], filtered[:, 2], "b--", lw=1.2, label="EKF Filtered")
axes[0].plot(smoothed[:, 0], smoothed[:, 2], "r-", lw=1.2, label="EKF Smoothed")
axes[0].set_xlabel("x (m)")
axes[0].set_ylabel("y (m)")
axes[0].set_title("Radar Target Tracking: Trajectory")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_aspect("equal")

# Position error plot
err_filt = np.sqrt((filtered[:, 0] - true_aircraft[:, 0])**2 +
                   (filtered[:, 2] - true_aircraft[:, 2])**2)
err_smooth = np.sqrt((smoothed[:, 0] - true_aircraft[:, 0])**2 +
                     (smoothed[:, 2] - true_aircraft[:, 2])**2)
t2 = np.arange(N2) * DT

axes[1].plot(t2, err_filt, "b-", alpha=0.7, label=f"Filtered (mean={err_filt.mean():.1f} m)")
axes[1].plot(t2, err_smooth, "r-", alpha=0.7, label=f"Smoothed (mean={err_smooth.mean():.1f} m)")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Position Error (m)")
axes[1].set_title("Filtering vs Smoothing Error")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean position error --- Filtered: {err_filt.mean():.1f} m, Smoothed: {err_smooth.mean():.1f} m")
print(f"Smoothing reduced error by {(1 - err_smooth.mean() / err_filt.mean()) * 100:.1f}%")

---
## Section 3: High-Dimensional System --- Ensemble Kalman Filter

When the state dimension is large (50+), the UKF requires $2n + 1$ sigma points and
full covariance matrices, which gets expensive fast. The **EnKF** uses a fixed-size
ensemble (e.g. 100 members) regardless of state dimension, and supports Dask-parallel
propagation.

We simulate a **coupled oscillator** chain with 50 state variables observed at every
5th point (10 sensors), and compare EnKF vs UKF runtime.

In [ ]:
# --- Coupled oscillator dynamics ---
n_state = 50
n_obs = 10
obs_indices = np.arange(0, n_state, 5)  # observe every 5th state

def f_chain(x):
    """Coupled oscillator: each state influenced by neighbors."""
    x_new = x.copy()
    x_new[1:-1] += 0.01 * (x[:-2] - 2 * x[1:-1] + x[2:])
    return x_new

def h_chain(x):
    """Observe every 5th state (10 out of 50)."""
    return x[obs_indices]

In [ ]:
# --- Generate ground truth ---
N3 = 200
Q3 = np.eye(n_state) * 0.001
R3 = np.eye(n_obs) * 0.1

true_chain = np.zeros((N3, n_state))
true_chain[0] = np.sin(np.linspace(0, 2 * np.pi, n_state))  # initial wave

meas_chain = np.zeros((N3, n_obs))

for k in range(1, N3):
    true_chain[k] = f_chain(true_chain[k - 1]) + np.random.multivariate_normal(
        np.zeros(n_state), Q3)

for k in range(N3):
    meas_chain[k] = h_chain(true_chain[k]) + np.random.multivariate_normal(
        np.zeros(n_obs), R3)

print(f"State dimension: {n_state}, Observation dimension: {n_obs}")
print(f"Timesteps: {N3}")

In [ ]:
# --- Fit EnKF (Dask disabled for fair timing comparison) ---
x0_chain = np.zeros(n_state)
P0_chain = np.eye(n_state) * 0.5

enkf = EnsembleKalmanFilter(f_chain, h_chain, Q3, R3, x0_chain, P0_chain,
                            n_ensemble=100, use_dask=False)

t_start = time.perf_counter()
enkf.fit(meas_chain)
t_enkf = time.perf_counter() - t_start
enkf_states = enkf.predict()

print(f"EnKF: {t_enkf:.3f} s")

In [ ]:
# --- Fit UKF on the same problem for comparison ---
ukf_hd = UnscentedKalmanFilter(f_chain, h_chain, Q3, R3, x0_chain, P0_chain,
                               alpha=1e-3, beta=2.0, kappa=0)

t_start = time.perf_counter()
ukf_hd.fit(meas_chain)
t_ukf = time.perf_counter() - t_start
ukf_hd_states = ukf_hd.predict()

print(f"UKF:  {t_ukf:.3f} s")

In [ ]:
# --- Timing and accuracy comparison ---
rmse_enkf = np.sqrt(np.mean((enkf_states - true_chain) ** 2))
rmse_ukf_hd = np.sqrt(np.mean((ukf_hd_states - true_chain) ** 2))

print(f"{'Filter':<8} {'RMSE':>10} {'Time (s)':>10} {'Sigma pts / Ensemble':>22}")
print("-" * 52)
print(f"{'EnKF':<8} {rmse_enkf:>10.5f} {t_enkf:>10.3f} {'100 members':>22}")
print(f"{'UKF':<8} {rmse_ukf_hd:>10.5f} {t_ukf:>10.3f} {f'{2*n_state+1} sigma pts':>22}")
print(f"\nUKF/EnKF time ratio: {t_ukf / t_enkf:.1f}x")

In [ ]:
# --- Plot selected states ---
states_to_plot = [0, 12, 24, 37, 49]

fig, axes = plt.subplots(len(states_to_plot), 1, figsize=(12, 10), sharex=True)

for i, si in enumerate(states_to_plot):
    axes[i].plot(true_chain[:, si], "k-", lw=1.5, label="True")
    axes[i].plot(enkf_states[:, si], "b--", lw=1, alpha=0.8, label="EnKF")
    axes[i].plot(ukf_hd_states[:, si], "r:", lw=1, alpha=0.8, label="UKF")
    observed = "(observed)" if si in obs_indices else "(unobserved)"
    axes[i].set_ylabel(f"State {si}")
    axes[i].set_title(f"State {si} {observed}", fontsize=10)
    axes[i].grid(True, alpha=0.3)
    if i == 0:
        axes[i].legend(loc="upper right", fontsize=9)

axes[-1].set_xlabel("Timestep")
fig.suptitle("High-Dimensional Coupled Oscillator: EnKF vs UKF", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## Summary

- **EKF** is fast and accurate when you can supply Jacobians. It also supports RTS smoothing.
- **UKF** achieves similar accuracy without Jacobians but scales poorly to high dimensions.
- **EnKF** handles high-dimensional systems efficiently with a fixed ensemble size and optional Dask parallelism.